# LAB 2 for CS4243: Perspective-Aware Virtual Advertisement Overlay
## Notebook 4 — Stage 4: your clip, the Overlay Scorecard, and the generative comparison
> **Introduction.** **Part I** applies the *frozen* pipeline to your one personal clip, compares the two temporal
> variants against your audit annotations, and assembles the Overlay Scorecard. **Part II** runs the generative
> comparison with an open-weight video model on your clip: one attempt with fixed settings and seed, kept and logged, its audit
> frames extracted and scored with the supplied `overlaylab.evaluate --frames`,
> side by side with your classical output on the same frames.
> Question 5 (the reflection) is written from the comparison table and the prompts at the end.

# Part I — Your clip and its Overlay Scorecard
## Suggested investigation structure
1. Describe the clip: board, capture conditions, camera pan / distance change / viewing-angle change, the occlusion
   crossing, with frame ranges (from `manifests/capture_manifest.csv`).
2. Record the audit annotations **before** running anything: 10 corner frames in `manifests/audit_corners.csv`, 5 masks
   (at least 3 on occluded frames) in `manifests/audit_masks.csv`.
   Use `python -m overlaylab.annotate corners | masks | check <clip_id>` (README, "Annotate your clip").
3. Run `python -m overlaylab.run_pipeline --clip <clip_id> --variant independent` and `--variant flow_assisted` with
   `--thresholds outputs/thresholds.json` and your own artwork (`--artwork <name>`), then
   `python -m overlaylab.evaluate --video outputs/videos/<clip_id>_<variant>.mp4 --clip <clip_id> --artwork <name>`.
4. Load the `_stats.json` files written next to the videos and compute corner error, success, jitter, drift and the
   occlusion IoU / F1 / leakage against your audit annotations through `overlaylab.metrics`.
5. Plot the monitor timeline around the occlusion crossing and fill the Scorecard.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np
import matplotlib.pyplot as plt
import cv2

from overlaylab import data, metrics, visualize as viz
from overlaylab import supplied as sp
from overlaylab.config import notebook_config, replace
from overlaylab.pipelines import load_stats
from overlaylab.temporal import failure_monitor
from overlaylab.evaluate import evaluate_video, evaluate_frames, format_report
import json

cond = data.load_genai_conditions()
CLIP_ID = cond["clip_id"] if not cond["clip_id"].startswith("<") else "my_clip"   # your clip id (data/personal/<clip_id>/), also set in manifests/genai_conditions.json
ARTWORK = Path(cond["artwork"]).stem if "<" not in cond["artwork"] else None
# Set your own artwork in genai_conditions.json; never silently substitute the supplied ad.
if ARTWORK is None: print("Set your own artwork in manifests/genai_conditions.json before Stage 4.")
from overlaylab.config import PipelineConfig
freeze_path = ROOT / "outputs" / "thresholds.json"
frozen = PipelineConfig.from_json(freeze_path.read_text()) if freeze_path.exists() else PipelineConfig()
SUCCESS_THRESHOLD = frozen.localisation.success_threshold
video, reference_path = data.clip_paths(CLIP_ID)
print("clip:", CLIP_ID, "| video exists:", video.exists(), "| artwork:", ARTWORK)
audit_c = data.load_audit_corners(CLIP_ID) if (data.MANIFEST_DIR / "audit_corners.csv").exists() else {}
audit_m = data.load_audit_masks(CLIP_ID) if (data.MANIFEST_DIR / "audit_masks.csv").exists() else {}
print(len(audit_c), "audit corner frames", sorted(audit_c), "|", len(audit_m), "audit masks", sorted(audit_m))
stats_path = ROOT / "outputs" / "videos" / f"{CLIP_ID}_flow_assisted_stats.json"
if stats_path.exists() and audit_c:
    st = load_stats(stats_path)
    scale = (st["resize_width"] or sp.video_info(video)["width"]) / sp.video_info(video)["width"]
    errs = {f: metrics.corner_error(np.asarray(st["corners"][f]) / scale, audit_c[f]) for f in audit_c if f < len(st["corners"])}
    print("corner errors on audit frames (%):", {f: round(e * 100, 2) for f, e in errs.items()})
    viz.plot_monitor_timeline(st["stats"], [tuple(x) for x in st["flags"]], errors=errs, title=CLIP_ID); plt.show()
else:
    print("run the pipeline on your clip first:", stats_path.name, "not found (or no audit corners yet)")

## Classical evaluation on the audit frames
`overlaylab.evaluate` scores both variants of your clip on your audit corners and masks (placement, fidelity, temporal,
preservation, occlusion incl. the visible-board coverage). These are the numbers for the Scorecard.

In [ ]:
classical = {}
for variant in ("independent", "flow_assisted"):
    path = ROOT / "outputs" / "videos" / f"{CLIP_ID}_{variant}.mp4"
    if path.exists() and audit_c and ARTWORK:
        classical[variant] = evaluate_video(path, video, data.load_artwork(ARTWORK), gt_corners=audit_c, gt_masks=audit_m, success_threshold=SUCCESS_THRESHOLD, verbose=False)
        print(format_report(classical[variant]), "\n")
        folder = ROOT / "outputs" / "evaluation"
        folder.mkdir(parents=True, exist_ok=True)
        (folder / f"{CLIP_ID}_{variant}_report.json").write_text(json.dumps(classical[variant], indent=2, default=float))
    else:
        print("missing:", path.name)

## Overlay Scorecard
Fill the dictionary from your measurements and write the Markdown scorecard to `outputs/scorecards/<clip_id>_scorecard.md`.
Every number must come from `overlaylab.metrics` or `overlaylab.evaluate`; the description and the uncertainty statement are yours.

In [ ]:
def write_scorecard(clip_id, fields):
    lines = [f"# Overlay Scorecard — {clip_id}", ""]
    for k, v in fields.items():
        lines.append(f"**{k}**: {v}  ")
    (ROOT / "outputs" / "scorecards").mkdir(parents=True, exist_ok=True)
    (ROOT / "outputs" / "scorecards" / f"{clip_id}_scorecard.md").write_text("\n".join(lines))
    return "\n".join(lines)

scorecard = {
    "Clip": "<conditions, resolution, fps, board size, event frame ranges>",
    "Initialisation": "<keypoints on reference / frame 0, matches, inliers (ratio), median residual>",
    "Geometry": "<median corner error and success fraction for both variants>",
    "Temporal": "<jitter and drift for both variants; recovery after re-entry>",
    "Occlusion": "<IoU, F1, leakage on masked frames; visible-board coverage; baseline leakage>",
    "Monitor": "<timeline: flags, frames and triggering statistics>",
    "Description": "<one evidence-grounded paragraph>",
    "Uncertainty": "<annotation uncertainty, thresholds transferred from the common clip, ...>",
}
# YOUR CODE HERE — replace the placeholders with measured values, then write the file
print(write_scorecard(CLIP_ID, scorecard))

# Part II — The generative comparison on your clip
## The prompt

The model receives your clip (or your cut of it), your artwork and a one-line instruction (plus your pipeline's board masks if it
needs a mask). This asks whether it can carry out the intended edit without explicit geometric or preservation
instructions.

Use **one open-weight video model** that returns the edited video itself (inference scripts are allowed; the edit itself must be generated pixels) and keep it, its version and
its settings (`model` in `manifests/genai_conditions.json`). Use a model your hardware supports, for example
MiniMax-H3 [MiniMax, 2026, https://huggingface.co/MiniMaxAI/MiniMax-H3], JoyAI-Video-Edit [JD Open Source, 2026,
https://github.com/jd-opensource/JoyAI-Video-Edit], LTX-2.3 [Lightricks, 2026] or Wan2.1-VACE-14B [Wan, arXiv:2503.20314;
VACE, Jiang et al., arXiv:2503.07598]; other open-weight models are equally welcome, and the grade rests on the comparison
rather than on the model. Wan2.1-VACE-1.3B (Apache 2.0; staff trial: 14 GB GPU memory and 13 min
for a 5.5 s cut at 480p) is a smaller option; memory use depends on settings. Cite the model you use. Give the model your
whole clip; if it does not accept a clip this long, **cut it**: a shorter interval that contains all of the frames below,
or consecutive segments run with the same settings whose outputs you join in order. **One attempt**, including any duration-limited segments, with fixed settings and one seed
(`model.seed`), its output kept as returned and logged in `manifests/genai_log.csv`, `[AI-GENERATION]` on every generated
pixel.

The comparison is scored on these audit frames: one **clean** frame (nothing in front of the board), one **occluded** frame (a mask
frame), the frame with the **strongest perspective**, and one consecutive **pair** (f, f+1) for the temporal terms.
`data/genai_oracle/` holds the same edit run by the staff on the common clip with MiniMax-H3 (33 B parameters) and its
report: an example of what a much larger model does, not a target.

## 1. Choose the frames (and a cut if your model needs one), export the inputs
Fill `frames` in `manifests/genai_conditions.json` **before the first generation**. Leave `model.interval` empty to give the
model the whole clip, or set it to the first and last frame of your cut. The cell exports to
`outputs/genai/inputs/`: the clip or cut as `interval.mp4` (source frame rate) and `masks/mask_XXXXXX.png`, the board quadrilateral your *frozen* pipeline tracked on every frame of the
interval (white = board), for models that take a mask. The audit annotations are never given to the model.

In [ ]:
from overlaylab.pipelines import load_stats
cond = data.load_genai_conditions(); fr = cond["frames"]; iv = cond["model"]["interval"]
assert not cond["clip_id"].startswith("<"), "set clip_id in manifests/genai_conditions.json"
assert all(isinstance(fr[k], int) for k in ("clean", "occluded", "perspective")), "fill frames.clean / occluded / perspective in genai_conditions.json"
assert isinstance(fr["pair"], list) and len(fr["pair"]) == 2 and all(isinstance(x, int) for x in fr["pair"]) and fr["pair"][1] == fr["pair"][0] + 1, "frames.pair must be two consecutive frame indices"
assert fr["occluded"] in audit_m and fr["occluded"] in audit_c, "the occluded frame needs both an audit mask and audit corners"
assert fr["clean"] in audit_c and fr["perspective"] in audit_c, "clean and perspective frames must be audit corner frames"
assert len({fr["clean"], fr["occluded"], fr["perspective"]}) == 3, "choose three distinct audit frames"
assert ARTWORK is not None, "set your own artwork first"
chosen = sorted({fr["clean"], fr["occluded"], fr["perspective"], *fr["pair"]})
info = sp.video_info(video)
iv = {"start": iv["start"] if isinstance(iv.get("start"), int) else 0,
      "end": iv["end"] if isinstance(iv.get("end"), int) else info["frames"] - 1}       # empty = the whole clip
assert 0 <= iv["start"] <= min(chosen) and max(chosen) <= iv["end"] < info["frames"], f"the cut {iv['start']}..{iv['end']} must contain the frames {chosen}"
print("model:", cond["model"].get("name"), "| interval", iv["start"], "-", iv["end"], f"({(iv['end'] - iv['start'] + 1) / info['fps']:.1f} s)", "| audit frames", chosen)
inputs = ROOT / "outputs" / "genai" / "inputs"; (inputs / "masks").mkdir(parents=True, exist_ok=True)
stats_path = ROOT / "outputs" / "videos" / f"{CLIP_ID}_flow_assisted_stats.json"
stats = load_stats(stats_path) if stats_path.exists() else None
if stats is None:
    print("  no", stats_path.name, "- run the frozen pipeline first (Part I) if your model needs board masks")
for stale in (inputs / "masks").glob("mask_*.png"): stale.unlink()
first = None
with sp.VideoWriter(inputs / "interval.mp4", info["fps"], (info["width"], info["height"])) as w:
    for i, f in sp.iter_video(video, start=iv["start"], stop=iv["end"] + 1):
        w.write(f)
        if first is None:
            first = (i, f)
        if stats is not None:
            c = stats["corners"][i]
            c = np.asarray(c, float) * info["width"] / (stats.get("resize_width") or info["width"])
            q = np.asarray(c, float) if c is not None else None
            ok = q is not None and q.shape == (4, 2) and np.all(np.isfinite(q))      # frames the pipeline lost carry NaN corners
            m = sp.quad_mask(f.shape[:2], q) if ok else np.zeros(f.shape[:2], bool)
            sp.write_mask(inputs / "masks" / f"mask_{i:06d}.png", m)
print("exported interval.mp4 and", len(list((inputs / "masks").glob("*.png"))), "masks to", inputs)
print("\ninputs:", cond["prompt"]["inputs"], "\nprompt:", cond["prompt"]["template"])
print("\nSave the model's video as outputs/genai/output.mp4, then run the next cell.")

## 2. Extract the audit frames from every output video
A video model resamples the clip to its own frame rate (16 fps for Wan, 24 fps for MiniMax-H3) and may re-time the motion.
Output frame *k* is taken to show source time `start + k · fps_source / fps_output`; the cell writes the output frame closest
to each audit frame's source time, resized to your clip's resolution, as `frame_XXXXXX.png` with the **original index**.
Set `model.output_fps` in `genai_conditions.json` if the container reports the wrong frame rate. If you split the clip into
segments, join the outputs in order into one `output.mp4` first. If the model changed the
timing, these frames will not line up with your clip, and the evaluator will count that as an error. Report it.

In [ ]:
gdir = ROOT / "outputs" / "genai"; mp4 = gdir / "output.mp4"
if not mp4.exists():
    print("no outputs/genai/output.mp4 yet")
else:
    oinfo = sp.video_info(mp4); ofps = cond["model"].get("output_fps") or oinfo["fps"]
    assert ofps > 0, "output fps must be positive"
    for stale in gdir.glob("frame_*.png"): stale.unlink()
    want = {f: int(round((f - iv["start"]) * ofps / info["fps"])) for f in chosen}
    if len(set(want.values())) != len(want):
        raise ValueError("output frame rate maps distinct selected source frames to one output frame; choose a consecutive pair that maps to two distinct frames before generation")
    got = {k: f for k, f in sp.iter_video(mp4) if k in set(want.values())}
    for f, k in want.items():
        if k not in got:
            raise ValueError(f"output has no frame {k} for required source frame {f}; do not score an incomplete comparison")
        out = cv2.resize(sp.to_uint8(got[k]), (info["width"], info["height"]), interpolation=cv2.INTER_CUBIC)
        sp.write_image(gdir / f"frame_{f:06d}.png", out)
    print(f"output.mp4: {oinfo['frames']} frames at {ofps:g} fps, {oinfo['width']}x{oinfo['height']} -> audit frames", {f: k for f, k in want.items()})

## 3. Log the run before looking at the metrics
One attempt in `manifests/genai_log.csv` (one row per segment/invocation if segmented; log technical retries too) (copy the template; the fields are `cond["log_fields"]`). Record generated pixels with
`[AI-GENERATION]`; state the model, visible version, date, settings, the seed, the video you gave it (whole clip, cut, or
joined segments), the output frame rate, the prompt, time and compute.

In [ ]:
import csv, shutil
log = ROOT / "manifests" / "genai_log.csv"
if not log.exists():
    shutil.copy(ROOT / "manifests" / "genai_log_template.csv", log)
rows = list(csv.DictReader(open(log)))
fields = list(csv.DictReader(open(log)).fieldnames)
print(len(rows), "invocation(s) logged for one attempt (multiple rows are allowed for segments/technical retries)")
missing_fields = [f for f in cond["log_fields"] if f not in fields]
print("fields:", fields, "| missing:", missing_fields or "none")
for r in rows:
    if not r.get("model", "").strip() or not r.get("seed", "").strip():
        print("  WARNING: the log row does not declare model / seed")

## 4. Evaluate the generative output and the classical output on the same frames
`overlaylab.evaluate --frames` re-localises the artwork in each generated frame with SIFT + RANSAC, compares the rectified
board with the artwork, checks what changed outside the board, measures leakage and visible-board coverage on the mask
frame, and the flicker / warping error / jitter on the consecutive pair. Both classical videos are extracted and scored with
`evaluate_frames` on the **same selected indices and full audit dictionary**, so every row of the table is computed on identical inputs.

In [ ]:
assert ARTWORK is not None, "set your own artwork"
art = data.load_artwork(ARTWORK)
# Both methods use the identical selected source indices and the full audit dictionary.
# This also gives both the same interpolation on the consecutive pair.
reports = {}
for variant in ("independent", "flow_assisted"):
    path = ROOT / "outputs" / "videos" / f"{CLIP_ID}_{variant}.mp4"
    assert path.exists(), f"run {variant} on your clip first"
    folder = ROOT / "outputs" / "genai" / f"classical_{variant}"
    folder.mkdir(parents=True, exist_ok=True)
    for stale in folder.glob("frame_*.png"): stale.unlink()
    for i, frame in sp.iter_video(path, stop=max(chosen) + 1):
        if i in chosen: sp.write_image(folder / f"frame_{i:06d}.png", frame)
    paths = [folder / f"frame_{f:06d}.png" for f in chosen]
    assert all(p.exists() for p in paths), "classical output is incomplete"
    reports[f"classical_{variant}"] = evaluate_frames(paths, video, art, gt_corners=audit_c, gt_masks=audit_m,
        success_threshold=SUCCESS_THRESHOLD, verbose=False)
paths = [ROOT / "outputs" / "genai" / f"frame_{f:06d}.png" for f in chosen]
assert all(p.exists() for p in paths), "extract every selected generative frame first"
reports["generative"] = evaluate_frames(paths, video, art, gt_corners=audit_c, gt_masks=audit_m,
    success_threshold=SUCCESS_THRESHOLD, verbose=False)
for name, rep in reports.items(): print(name, format_report(rep), "\n")
(ROOT / "outputs" / "evaluation").mkdir(exist_ok=True)
(ROOT / "outputs" / "evaluation" / "stage4_genai_reports.json").write_text(json.dumps(reports, indent=2, default=float))

for name, rep in reports.items():
    fname = "genai_frames_report.json" if name == "generative" else f"{name}_frames_report.json"
    (ROOT / "outputs" / "evaluation" / fname).write_text(json.dumps(rep, indent=2, default=float))


## 5. Comparison table
Placement, fidelity, temporal consistency, preservation and occlusion for the generative output next to the classical
output on the same frames, plus processing time and compute from your log. The background PSNR is the signature
of what the system did: pixels *composited by code* leave the background unchanged up to compression (about 40 dB: your
pipeline), a *re-rendered* frame scores about 20-30 dB (typically the generative model).

In [ ]:
# YOUR CODE HERE — complete the table from `reports` and your log (processing_time_s, compute)
def total_seconds(log_rows):
    vals = [r.get("processing_time_s", "").strip() for r in log_rows]
    return str(sum(float(v) for v in vals if v)) if any(vals) else ""
by_path = {"generative": {"processing_time_s": total_seconds(rows),
                          "compute": "; ".join(dict.fromkeys(r.get("compute", "") for r in rows))}} if rows else {}
if reports:
    print(f"{'output':26s} {'median err':>10s} {'success':>8s} {'NCC':>6s} {'flicker':>8s} {'warp err':>9s} {'bg PSNR':>8s} {'leakage':>8s} {'coverage':>9s} {'time s':>7s} {'compute':>8s}")
    for name, r in reports.items():
        p, f, t, s, o = r["placement"], r["fidelity"], r["temporal"], r["preservation"], r["occlusion"]
        g = lambda v, fmt: fmt.format(v) if v is not None and np.isfinite(v) else "n/a"
        lg = by_path.get(name, {})
        print(f"{name:26s} {g(p['median'], '{:9.2%}')} {g(p['success'], '{:8.0%}')} {g(f.get('ncc', float('nan')), '{:6.3f}')} {g(t['flicker'], '{:8.4f}')} {g(t['warping_error'], '{:9.4f}')} "
              f"{g(s.get('psnr_outside', float('nan')), '{:7.1f}')} {g(o['leakage'], '{:8.1%}')} {g(o['coverage_visible'], '{:9.1%}')} {lg.get('processing_time_s', ''):>7s} {lg.get('compute', ''):>8s}")

### The staff oracle: MiniMax-H3 on the common clip
`data/genai_oracle/` holds one run of the same edit by a 33-billion-parameter open video model on the common clip, scored
by the same evaluator. Compare its failure modes with your model's: what does a much larger model fix, and what does it still get
wrong (background re-rendering, timing, occlusion order)?

In [ ]:
oracle = data.DATA_DIR / "genai_oracle"
if (oracle / "summary.md").exists():
    print((oracle / "summary.md").read_text())
    sheet = cv2.cvtColor(cv2.imread(str(oracle / "contact_sheet.jpg")), cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(16, 16 * sheet.shape[0] / sheet.shape[1])); plt.imshow(sheet); plt.axis("off")
    plt.title("rows: audit frames 160 / 192 / 320; columns: original, MiniMax-H3"); plt.show()
else:
    print("data/genai_oracle/ not found")

## 6. Reflection prompts for Question 5
Write the reflection from the table and the frames themselves. Classical versus generative on **your** clip along:

1. **Geometric placement** — does the artwork sit on the board's quadrilateral (median corner error, success) on the clean
   frame and on the strongest-perspective frame? Which method respects the keystone?
2. **Artwork identity** — is the artwork reproduced exactly (rectified NCC / SSIM / structure)? Where does the generative
   output drift (text, colours, layout)?
3. **Occlusion order** — on the occluded frame, does the occluder stay in front (leakage) and is the visible board painted
   (coverage)? Compare with your Stage 3 mask.
4. **Background preservation** — what does the background PSNR / SSIM / drift tell you about whether the pixels outside the
   board were *generated* (re-rendered frame) or *composited by code* (untouched)? Which method can you prove did not touch
   the background?
5. **Controllability** — what could you change by editing the prompt, what only by editing code?
6. **Timing** — does the output keep the clip's timing, or do the extracted audit frames no longer match their source frames
   (flicker and warping error on the consecutive pair, background drift)?
7. **Cost** — processing time, compute and manual effort, against the classical pipeline's run time.

Every claim must point to a row of the table or to a frame in `outputs/genai/`; mark generated pixels `[AI-GENERATION]`.

## Analysis ideas
* Plot corner error against mean flow magnitude for your clip; group frames by whether the monitor fired.
* Compare the two temporal variants frame by frame around the occlusion crossing.
* For the generative outputs, show one frame where the artwork identity drifts and quantify it with the rectified NCC.
* Treat the generation as a qualitative probe: one attempt says what the model *can* do, not how often.